# CS604 Colab launcher

Runtime: **GPU**. Images stay on Colab disk (`/content/data`). Checkpoints and logs go to the shared Drive folder **CS604_RUNS**.

The GitHub repo is private, so the clone cell will ask for a Personal Access Token (hidden). Do not paste the token into the notebook text or commit it.

**Drive:** everyone must have `CS604_RUNS` under *My Drive* (create it there, or open the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR) → Organize → **Add shortcut to Drive** → My Drive, name it `CS604_RUNS`). A Drive *link* is not a file path Colab can write to.

Change `METHOD`, `TEAM`, and `BRANCH` below before you run.

**Optional pilot sweep (5a):** if `RUN_PILOT = True`, short runs on a 1,000-class subset test each value in the config's `pilot.sweep` block, and the train step uses the winner. The train step also sets `epochs` from the pilot's measured speed so training fits `BUDGET_MIN`. With `RUN_PILOT = False` the launcher behaves as before.

In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "ana-branch"  # must contain scripts/pilot_sweep.py and the method config
METHOD = "full_ft_336"  # linear_probe | full_ft | full_ft_336 | lora_r8 | bitfit | last_blocks | prototype_probe | prototype_lora_r8
TEAM = "08"
PROTO_PT = f"{DRIVE_OUT}/prototypes.pt"

# Pilot sweep + time budget (optional)
RUN_PILOT = True  # needs a pilot: block in configs/{METHOD}.yaml
PILOT_DIR = ""    # paste an earlier pilot folder on Drive to reuse it instead of rerunning
PILOT_DATA_DIR = "/content/pilot_data"  # local disk; label files only, images are symlinked
BUDGET_MIN = 144  # 2 hr x 48GB, taken as 144 min on an A100 40GB
SAFETY = 0.95     # plan epochs to use at most 95% of the budget

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

## 4b. Build class prototypes (once per backbone)

Van's ~43.9% zero-shot head. Run **once**, then reuse `PROTO_PT` for `prototype_probe` / `prototype_lora_r8`. Takes a while (embeds train images). Skip if the file already exists on Drive.

In [ ]:
from pathlib import Path

if Path(PROTO_PT).is_file():
    print("Already have", PROTO_PT)
else:
    !python scripts/build_prototypes.py --config configs/prototype_probe.yaml --data_root {DATA_DIR} --output {PROTO_PT}

## Review settings (table)

Loads `configs/{METHOD}.yaml` (merged with `base.yaml`) so you can check every parameter before spending a GPU hour.

In [ ]:
import pandas as pd

from src.utils import apply_overrides, load_config

methods = ["full_ft", "linear_probe", "lora_r8", "bitfit", "last_blocks", "prototype_probe", "prototype_lora_r8", "adapter", "adaptformer"]
if METHOD not in methods:
    methods.append(METHOD)
merged = {name: load_config(f"configs/{name}.yaml").to_dict() for name in methods}
keys = []
for d in merged.values():
    for k in d:
        if k not in keys:
            keys.append(k)

compare = pd.DataFrame(
    {name: [merged[name].get(k, "") for k in keys] for name in methods},
    index=keys,
)
compare.index.name = "parameter"
print("Resolved parameters per method (base.yaml already merged in):")
display(compare)

chosen = load_config(f"configs/{METHOD}.yaml")
apply_overrides(chosen, data_root=DATA_DIR, output_dir=DRIVE_OUT)
resolved = pd.DataFrame(list(chosen.to_dict().items()), columns=["parameter", "value"])
print(f"\nThis run (METHOD={METHOD!r}, plus Colab data_root / output_dir):")
display(resolved)

## 5a. Pilot sweep (optional)

Runs only if `RUN_PILOT = True`. Each value in `pilot.sweep` gets a short run (default: 1,000 classes, 3 epochs) scored on a labelled 5-per-class holdout. Writes `pilot_sweep.csv`, `pilot_summary.json` and `best_config.yaml` (the full-run config with the chosen values) to a `pilot_*` folder on Drive. About 6 to 7 min per value at 336 on an A100.

Check the curves before training: an accuracy dip partway through means the lr is too high, and a winner at the edge of the sweep means the sweep should be extended.

In [ ]:
import json

import matplotlib.pyplot as plt

TRAIN_CONFIG = f"configs/{METHOD}.yaml"
summary = None
if RUN_PILOT:
    if PILOT_DIR:
        sweep_dir = Path(PILOT_DIR)
        print("Reusing pilot:", sweep_dir)
    else:
        !python scripts/pilot_sweep.py --config configs/{METHOD}.yaml --data_root {DATA_DIR} \
            --output_dir {DRIVE_OUT} --pilot_data_dir {PILOT_DATA_DIR}
        sweep_dir = sorted(Path(DRIVE_OUT).glob("pilot_*"), key=lambda p: p.stat().st_mtime)[-1]
        print("Pilot folder:", sweep_dir)
    display(pd.read_csv(sweep_dir / "pilot_sweep.csv"))
    summary = json.loads((sweep_dir / "pilot_summary.json").read_text())
    print("Chosen:", summary["chosen"], "|", summary["reason"])
    TRAIN_CONFIG = str(sweep_dir / "best_config.yaml")

    fig, ax = plt.subplots(figsize=(6, 4))
    for r in summary["results"]:
        log = pd.read_csv(Path(r["run_dir"]) / "epoch_log.csv")
        label = ", ".join(f"{k}={r[k]}" for k in summary["chosen"])
        ax.plot(log["epoch"], log["val_acc"], marker="o", label=label)
    ax.set_xlabel("epoch"); ax.set_ylabel("pilot holdout top-1"); ax.legend(); ax.grid(alpha=0.3)
    plt.show()
else:
    print("No pilot; training with", TRAIN_CONFIG)

## 5. Train (checkpoints on Drive)

Uses `TRAIN_CONFIG` from 5a (the pilot's `best_config.yaml`, or `configs/{METHOD}.yaml` without a pilot).

**Time budget:** with a pilot, `epochs` is set from the pilot's measured speed so training fits `SAFETY × BUDGET_MIN`, keeping the cosine schedule complete. Without one, the config's `epochs` is used, so watch the first epoch's time. The run is wrapped in `timeout` as a safety net; `last.pt` is saved every epoch, but an interrupted run should be treated as failed and rerun with fewer epochs.

In [ ]:
import math

train_cfg = load_config(TRAIN_CONFIG)
n_train = sum(1 for _ in open(Path(DATA_DIR) / "train.txt"))
epochs = int(train_cfg.epochs)

if summary is not None:
    epoch_times = []
    for r in summary["results"]:
        log = pd.read_csv(Path(r["run_dir"]) / "epoch_log.csv")
        # skip epoch 1 when possible: it includes first-pass disk reads and warm-up
        times = log["epoch_time_s"].iloc[1:] if len(log) > 1 else log["epoch_time_s"]
        epoch_times.extend(times.tolist())
    img_per_s = summary["pilot_data"]["fit_images"] / float(pd.Series(epoch_times).median())
    min_per_epoch = n_train / img_per_s / 60
    fit_epochs = max(1, math.floor(SAFETY * BUDGET_MIN / min_per_epoch))
    print(f"Pilot speed: {img_per_s:.0f} img/s -> {min_per_epoch:.1f} min per full epoch ({n_train:,} images)")
    print(f"Config epochs: {epochs}; fits in {SAFETY:.0%} of {BUDGET_MIN} min: {fit_epochs}")
    epochs = min(epochs, fit_epochs)
    print(f"Planned: {epochs} epochs, about {epochs * min_per_epoch:.0f} min of training")
else:
    print(f"No pilot timing; using config epochs={epochs}.")

HARD_STOP_MIN = BUDGET_MIN + 5  # small allowance for model loading
!timeout --signal=INT {HARD_STOP_MIN}m python -m src.train --config {TRAIN_CONFIG} \
    --data_root {DATA_DIR} --output_dir {DRIVE_OUT} --epochs {epochs}

In [ ]:
run_dir = sorted(Path(DRIVE_OUT).glob(f"{train_cfg.method}_*"), key=lambda p: p.stat().st_mtime)[-1]
print("Run folder:", run_dir)
log = pd.read_csv(run_dir / "epoch_log.csv")
display(log)

train_min = log["epoch_time_s"].sum() / 60
print(f"Training time: {train_min:.1f} min of {BUDGET_MIN} min budget ({train_min / BUDGET_MIN:.0%})")
if train_min > BUDGET_MIN:
    print("WARNING: over budget. Reduce epochs and rerun.")
if len(log) < epochs or not (run_dir / "run_stats.txt").exists():
    print(f"WARNING: run stopped early ({len(log)}/{epochs} epochs); likely hit the hard stop.")
else:
    print((run_dir / "run_stats.txt").read_text())

## 6. Predict

`CKPT` defaults to the run from step 5 (`best.pt` if it exists, otherwise `last.pt`; the course val split has no labels, so full-data runs only have `last.pt`). To predict from another run, set `CKPT` by hand.

In [ ]:
CKPT = run_dir / ("best.pt" if (run_dir / "best.pt").exists() else "last.pt")
PRED_CONFIG = run_dir / "config.yaml"  # the exact config that was trained
print("Checkpoint:", CKPT)

!python -m src.predict --config {PRED_CONFIG} --ckpt {CKPT} --split val --team {TEAM} --data_root {DATA_DIR} --out_dir {DRIVE_OUT}
!python -m src.predict --config {PRED_CONFIG} --ckpt {CKPT} --split test --team {TEAM} --data_root {DATA_DIR} --out_dir {DRIVE_OUT}